In [1]:
# =====================================================
# FINAL LPBF HAYNES 282 DATASET GENERATOR
# Uses melt pool ML mini-model
# =====================================================

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

In [2]:
# -----------------------------
# CONFIG
# -----------------------------
N_SAMPLES = 500
SEED = 42
np.random.seed(SEED)

In [3]:
# -----------------------------
# MATERIAL CONSTANTS (Haynes 282)
# -----------------------------
Cp = 436.0                # J/kg-K
k = 10.3                  # W/m-K
k_mm = 0.0103             # W/mm-K
rho = 8300                # kg/m^3
Tm = 1623                 # K
T0 = 300                  # K
eta = 0.40                # absorptivity


In [4]:
# -----------------------------
# PROCESS WINDOW
# -----------------------------
P = np.random.randint(150, 371, N_SAMPLES)      # 150 to 370 inclusive (W)
v = np.random.randint(350, 1773, N_SAMPLES)     # 350 to 1772 inclusive (mm/s)
h = np.random.choice([0.09,0.10,0.11,0.12], N_SAMPLES)    # mm
t = np.random.choice([0.03, 0.04, 0.05, 0.06], N_SAMPLES)    # mm

# convert speed for melt pool model
v_mps = v / 1000.0

In [5]:
# =====================================================
# 1. ENERGY FEATURES
# =====================================================

VED = P / (v * h * t)
LED = P / v

Pi1 = (Cp * P) / (k_mm * (v**2) * h)

In [6]:
# =====================================================
# 2. MELT POOL MODEL (FITTED APPROACH)
# =====================================================

# -----------------------------------------
# ORIGINAL EXPERIMENTAL DATA
# -----------------------------------------
train_data = pd.DataFrame({
    "P": [150,150,150,150,150,250,250,250,250,250,250,250,350,350,350,350,350,350,350,370],
    "V": [350,553,756,959,1163,350,553,756,959,1163,1366,1772,350,553,756,959,1163,1366,1772,1772],
    "W": [201,166,134,122,108,223,193,173,149,127,126,130,242,181,186,167,144,130,129,134],
    "D": [177,91,57,50,40,336,190,132,103,83,70,50,492,304,199,152,121,106,80,85]
})

train_data["V"] = train_data["V"] / 1000

# =====================================================
# WIDTH MODEL (LOG-LINEAR)
# =====================================================

train_data["log_val"] = np.log(train_data["P"] / train_data["V"])

Xw = train_data[["log_val"]]
yw = np.log(train_data["W"])

model_W = LinearRegression()
model_W.fit(Xw, yw)

# Predict Width
X_pred_w = np.log(P / v_mps).reshape(-1,1)
W = np.exp(model_W.predict(X_pred_w))

# Add realistic noise
W += np.random.normal(0, 6, N_SAMPLES)

# =====================================================
# DEPTH MODEL (POLYNOMIAL)
# =====================================================

train_data["P_over_V"] = train_data["P"] / train_data["V"]
train_data["P_over_V2"] = train_data["P"] / (train_data["V"]**2)

Xd = train_data[["P","V","P_over_V","P_over_V2"]]
yd = train_data["D"]

poly = PolynomialFeatures(degree=2, include_bias=False)
Xd_poly = poly.fit_transform(Xd)

model_D = LinearRegression()
model_D.fit(Xd_poly, yd)

# Predict Depth
X_pred_d = pd.DataFrame({
    "P": P,
    "V": v_mps,
    "P_over_V": P/v_mps,
    "P_over_V2": P/(v_mps**2)
})

X_pred_d_poly = poly.transform(X_pred_d)

D = model_D.predict(X_pred_d_poly)

# Add realistic noise
D += np.random.normal(0, 10, N_SAMPLES)

# Physical clipping
W = np.clip(W, 80, 280)
D = np.clip(D, 35, 520)


c:\Users\satya\anaconda3\envs\ml_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [7]:
# =====================================================
# 5. RELATIVE DENSITY
# EXPERIMENTALLY CALIBRATED RIDGE MINI-MODEL
# =====================================================

# -----------------------------------------------------
# Load density mini-model
# -----------------------------------------------------

density_model = joblib.load(
    "RelativeDensity_mini_model.pkl"
)

# Load experimentally derived residual information
density_error_info = joblib.load(
    "RelativeDensity_error_model.pkl"
)

residual_pool = density_error_info[
    "residual_pool_pct"
]


# -----------------------------------------------------
# Input features for density model
# -----------------------------------------------------

X_property = pd.DataFrame({

    "P_W": P,
    "v_mm_per_s": v,
    "h_mm": h,
    "t_mm": t
})


# -----------------------------------------------------
# Predict density
# -----------------------------------------------------

density_pct = density_model.predict(
    X_property
)

# Model predicts density deficit,
# so convert:
#
# density deficit → relative density

density_pct = (
    1.0 - density_pct
) * 100.0


# -----------------------------------------------------
# Add experimentally derived residual variation
# -----------------------------------------------------

density_error = np.random.choice(
    residual_pool,
    size=N_SAMPLES,
    replace=True
)

density_pct = (
    density_pct
    + density_error
)


# -----------------------------------------------------
# Physical constraint
# Relative density MUST be <= 100%
# -----------------------------------------------------

density_pct = np.clip(
    density_pct,
    0.0,
    100.0
)


# -----------------------------------------------------
# Convert percentage → fraction
# -----------------------------------------------------

density = (
    density_pct / 100.0
)


# -----------------------------------------------------
# Porosity derived internally
# -----------------------------------------------------

porosity = (
    1.0 - density
)

In [8]:
# =====================================================
# 6. DEFECT LABELS
# =====================================================

defect = np.zeros(N_SAMPLES, dtype=int)

# 0 = LOF
# 1 = Optimal
# 2 = Keyhole

defect[Pi1 < 61] = 0
defect[(Pi1 >= 61) & (Pi1 <= 146)] = 1
defect[Pi1 > 146] = 2

In [12]:
# =====================================================
# UTS MINI-MODEL
# EXPERIMENTALLY CALIBRATED RIDGE MODEL
# =====================================================

uts_model = joblib.load(
    "UTS_mini_model.pkl"
)

uts_error_info = joblib.load(
    "UTS_calibration.pkl"
)

uts_sigma = uts_error_info[
    "selected_error_sigma_MPa"
]

# =====================================================
# 9. UTS
# EXPERIMENTALLY CALIBRATED UTS MINI-MODEL
# =====================================================

# Base UTS prediction from experimental mini-model
UTS_base = uts_model.predict(
    X_property
)

# Add experimentally calibrated residual variation
UTS_error = np.random.normal(
    0,
    uts_sigma,
    N_SAMPLES
)

UTS = (
    UTS_base
    + UTS_error
)

In [13]:
# =====================================================
# 10. FINAL DATAFRAME
# =====================================================

df = pd.DataFrame({

    # =========================
    # PROCESS INPUTS
    # =========================
    "P_W": P,
    "v_mm_per_s": v,
    "h_mm": h,
    "t_mm": t,

    # =========================
    # ENERGY FEATURES
    # =========================
    "VED_J_per_mm3": VED,
    "LED_J_per_mm": LED,
    "Pi1_dimensionless": Pi1,

    # =========================
    # MELT POOL FEATURES
    # =========================
    "MeltPoolWidth_um": W,
    "MeltPoolDepth_um": D,

    # =========================
    # DEFECT / DENSITY
    # =========================
    "RelativeDensity_frac": density,
    "DefectClass": defect,

    # =========================
    # MECHANICAL PROPERTY
    # =========================
    "UTS_MPa": UTS
})

In [14]:
# =====================================================
# SAVE
# =====================================================

df.to_csv(
    "haynes282_final_dataset.csv",
    index=False
)

print("Dataset created successfully!")

print(df.head())

print("\nShape:", df.shape)

print("\nDataset statistics:")
print(df.describe())

print("\nDefect class distribution:")
print(df["DefectClass"].value_counts().sort_index())

print("\nMean relative density by defect class:")
print(
    df.groupby("DefectClass")[
        "RelativeDensity_frac"
    ].mean()
)

print("\nMean UTS by defect class:")
print(
    df.groupby("DefectClass")[
        "UTS_MPa"
    ].mean()
)

print(
    "\nNumber of Pi1 < 40:",
    (df["Pi1_dimensionless"] < 40).sum()
)

Dataset created successfully!
   P_W  v_mm_per_s  h_mm  t_mm  VED_J_per_mm3  LED_J_per_mm  \
0  252         962  0.10  0.04      65.488565      0.261954   
1  329        1342  0.11  0.06      37.144922      0.245156   
2  242         613  0.09  0.06      73.107365      0.394780   
3  164        1426  0.09  0.04      31.946392      0.115007   
4  256        1353  0.11  0.05      34.401666      0.189209   

   Pi1_dimensionless  MeltPoolWidth_um  MeltPoolDepth_um  \
0         115.265586        144.776805        122.808348   
1          70.298725        150.307719         94.700861   
2         302.901324        176.697408        169.157264   
3          37.932507        119.143058         40.765305   
4          53.814703        134.052863         87.698692   

   RelativeDensity_frac  DefectClass     UTS_MPa  
0              1.000000            1  918.963938  
1              1.000000            1  856.761360  
2              1.000000            2  824.853519  
3              0.989926   

In [15]:
# =====================================================
# UTS DISTRIBUTION CHECK
# =====================================================

print("\n" + "=" * 70)
print("UTS DISTRIBUTION CHECK")
print("=" * 70)

print(f"Base UTS minimum : {UTS_base.min():.2f} MPa")
print(f"Base UTS maximum : {UTS_base.max():.2f} MPa")
print(f"Base UTS mean    : {UTS_base.mean():.2f} MPa")

print(f"\nFinal UTS minimum: {UTS.min():.2f} MPa")
print(f"Final UTS maximum: {UTS.max():.2f} MPa")
print(f"Final UTS mean   : {UTS.mean():.2f} MPa")
print(f"Final UTS std    : {UTS.std():.2f} MPa")

print("\nUTS percentiles:")
print(
    np.percentile(
        UTS,
        [1, 5, 25, 50, 75, 95, 99]
    )
)

print("\nNumber below experimental minimum (793 MPa):",
      (UTS < 793).sum())

print("Number above experimental maximum (939 MPa):",
      (UTS > 939).sum())


UTS DISTRIBUTION CHECK
Base UTS minimum : 725.30 MPa
Base UTS maximum : 980.60 MPa
Base UTS mean    : 852.83 MPa

Final UTS minimum: 704.98 MPa
Final UTS maximum: 1029.87 MPa
Final UTS mean   : 853.53 MPa
Final UTS std    : 63.84 MPa

UTS percentiles:
[713.6314223  749.86289229 807.84714424 854.63763715 898.25787651
 954.82242915 995.15048674]

Number below experimental minimum (793 MPa): 97
Number above experimental maximum (939 MPa): 42
